# 05. Ensemble & Submission

Combine all models, optimize weights, generate submission.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss
from pathlib import Path

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

y_train = train_df['target'].values

In [ ]:
# Load predictions
oof_preds = {}
test_preds = {}

for name in ['tfidf', 'sbert', 'deberta']:
    oof_path = OUTPUT_DIR / f'{name}_oof.npy'
    test_path = OUTPUT_DIR / f'{name}_test.npy'
    if oof_path.exists() and test_path.exists():
        oof_preds[name] = np.load(oof_path)
        test_preds[name] = np.load(test_path)
        loss = log_loss(y_train, oof_preds[name])
        print(f'{name}: OOF log_loss={loss:.4f}')

In [ ]:
# Grid search weights
best_loss = 999
best_weights = None

from itertools import product

names = list(oof_preds.keys())
print(f'Models: {names}')

for w_vals in product(np.arange(0, 1.1, 0.1), repeat=len(names)):
    if sum(w_vals) == 0:
        continue
    w_sum = sum(w_vals)
    w_normalized = [w / w_sum for w in w_vals]

    ens_oof = np.zeros_like(oof_preds[names[0]])
    for name, w in zip(names, w_normalized):
        ens_oof += w * oof_preds[name]

    loss = log_loss(y_train, ens_oof)
    if loss < best_loss:
        best_loss = loss
        best_weights = dict(zip(names, w_normalized))

print(f'\nBest OOF log_loss: {best_loss:.4f}')
print(f'Best weights: {best_weights}')

In [ ]:
# Generate final predictions
final_preds = np.zeros_like(test_preds[names[0]])
for name, w in best_weights.items():
    final_preds += w * test_preds[name]

# Normalize
final_preds = final_preds / final_preds.sum(axis=1, keepdims=True)
print(f'Final predictions shape: {final_preds.shape}')
print(f'Row sums: {final_preds.sum(axis=1)[:5]}')

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': final_preds[:, 0],
    'winner_model_b': final_preds[:, 1],
    'winner_tie': final_preds[:, 2],
})

submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'Saved submission to {OUTPUT_DIR / "submission.csv"}')
print(submission.head())

In [ ]:
# Verify
row_sums = submission[['winner_model_a', 'winner_model_b', 'winner_tie']].sum(axis=1)
print(f'Row sums: min={row_sums.min():.6f}, max={row_sums.max():.6f}')

# All probabilities valid
assert (final_preds >= 0).all() and (final_preds <= 1).all()
print('Validation passed')

In [ ]:
# Submit to Kaggle (uncomment when ready)
# !kaggle competitions submit llm-classification-finetuning -f output/submission.csv -m "TF-IDF + SBERT + DeBERTa ensemble"